# In Class Activity 04/09 

In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import sweetviz as sv

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, recall_score

importing in the dataset

In [2]:
CC_df = pd.read_csv("creditcard.csv")
CC_df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,0,55773,55917,51389,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,-1,140,3230,3011,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,0,59710,49986,104390,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,0,280913,283222,273160,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,-2,1512,2458,664,1814,0,0,1000,664,1500,0,0,0,0


initial EDA

In [3]:
report = sv.analyze(CC_df)
report.show_html("Credit_Card.html")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)


Report Credit_Card.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Base RandomForest

In [4]:
X = CC_df.drop("default payment next month", axis = 1)
y = CC_df["default payment next month"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)


rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))
accuracy_score(y_test, y_pred_rf)

              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800



0.8114583333333333

Base XGBoosting

In [5]:
XGB = XGBClassifier()
XGB.fit(X_train, y_train)

y_pred_XGB = XGB.predict(X_test)
print(classification_report(y_test, y_pred_XGB))
accuracy_score(y_test, y_pred_XGB)

              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800



0.8102083333333333

Cross Validated Models

In [6]:
base_cv_rf= cross_val_score(rf,X,y, cv = 5, scoring = "r2")
base_cv_xgb = cross_val_score(XGB, X,y, cv =5, scoring = "r2")

print(f'Baseline random forest CV R2 Scores: {base_cv_rf}')
print(f'Baseline xgb CV R2 Scores: {base_cv_xgb}')  

Baseline random forest CV R2 Scores: [-0.08896243 -0.07855495 -0.07371838 -0.10757437 -0.05195382]
Baseline xgb CV R2 Scores: [-0.10469189 -0.09790123 -0.07734581 -0.10152866 -0.05074468]


repeat baseline but with recall as eval method 

In [7]:
rf_cv_recall = cross_val_score(rf, X,y, cv = 5, scoring = "recall")
XGB_cv_recall = cross_val_score(XGB, X,y, cv = 5, scoring = "recall")

print(f'random forest CV recall Scores: {rf_cv_recall}')
print(f'xgb CV recall Scores: {XGB_cv_recall}')  

random forest CV recall Scores: [0.35532516 0.36158192 0.3653484  0.36346516 0.36629002]
xgb CV recall Scores: [0.35626767 0.37288136 0.37193974 0.36817326 0.37382298]


rebuild rf and xgb modeles with adjustment to class weights to adress class inbalance

In [8]:
rf_weighted = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

neg = np.sum(y == 0)
pos = np.sum(y == 1)

XGB_weighted = XGBClassifier(random_state = 42, scale_pos_weight = neg/pos)

In [9]:
rf_weighted.fit(X_train, y_train)
XGB_weighted.fit(X_train,y_train)

rf_weight_pred = rf_weighted.predict(X_test)
XGB_weight_pred = XGB_weighted.predict(X_test)

In [10]:
print(classification_report(y_test, rf_weight_pred))
accuracy_score(y_test, rf_weight_pred)

              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.65      0.36      0.46      1062

    accuracy                           0.82      4800
   macro avg       0.74      0.65      0.68      4800
weighted avg       0.80      0.82      0.79      4800



0.8154166666666667

In [11]:
print(classification_report(y_test, XGB_weight_pred))
accuracy_score(y_test, XGB_weight_pred)

              precision    recall  f1-score   support

           0       0.87      0.82      0.84      3738
           1       0.47      0.58      0.52      1062

    accuracy                           0.76      4800
   macro avg       0.67      0.70      0.68      4800
weighted avg       0.79      0.76      0.77      4800



0.7645833333333333

tuning using grid search

In [ ]:
param_grid_rf = {
    "n_estimators": [100,200],
    "max_depth": [None, 10]
}

rf_grid_search = GridSearchCV(rf, param_grid_rf, cv = 5, scoring = recall)